# NF-v3 preparation and relabelling

This notebook never modifies the raw CSV. It creates new artefacts in `nids-fair-retrain` and runs the repository's versioned script. Before running it, adjust only the paths in the next cell.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

# NEW folder: corrected data, manifests, graphs, and results only.
PROJECT_ROOT = Path('/content/drive/MyDrive/nids-fair-retrain')
# Repository cloned by this Colab session.
REPO_ROOT = Path('/content/temporalgnn-nids')
# Existing raw CSV: never written or moved.
INPUT_CSVS = [
    Path('/content/drive/MyDrive/nids-mitre/data/cicids2018-v3/f78acbaa2afe1595_NFV3DATA-A11964_A11964/data/NF-CICIDS2018-v3.csv'),
]
OUTPUT_DIR = PROJECT_ROOT / 'corrected_data' / 'infiltration_v1'

for directory in [PROJECT_ROOT, OUTPUT_DIR, PROJECT_ROOT / 'graphs', PROJECT_ROOT / 'results']:
    directory.mkdir(parents=True, exist_ok=True)

assert REPO_ROOT.is_dir(), f'Repository not found: {REPO_ROOT}'
for path in INPUT_CSVS:
    assert path.is_file(), f'Raw CSV not found: {path}'

print(f'Project: {PROJECT_ROOT}')
print(f'Output: {OUTPUT_DIR}')
print('Inputs:', *INPUT_CSVS, sep='\n- ')

## Optional: validate the Category-0 direction mapping

Run this section only if you have a CICFlowMeter CSV that retains source/destination IPs and ports. A CSV containing only aggregate features and timestamps cannot establish a trustworthy direction mapping. The cell compares `Total Length of Fwd Packets` against NF-v3 `IN_BYTES` and `OUT_BYTES` for matched Dropbox and victim-attacker flows.

In [ ]:
# Set this to a CICFlowMeter CSV with endpoint columns. Leave as None to skip validation.
CIC_CSV = None  # Path('/content/drive/MyDrive/nids-mitre/data/cicids2018/...csv')

# Change values only if your CICFlowMeter export uses different column names.
CIC_COLUMNS = {
    'time': 'Timestamp',
    'source_ip': 'Src IP',
    'destination_ip': 'Dst IP',
    'source_port': 'Src Port',
    'destination_port': 'Dst Port',
    'forward_bytes': 'Total Length of Fwd Packets',
}
MATCH_TOLERANCE = '2s'
# Use the timezone represented by a naive CICFlowMeter Timestamp.
CIC_TIMESTAMP_TIMEZONE = 'UTC'


In [ ]:
import pandas as pd

if CIC_CSV is None:
    print('Set CIC_CSV to run the direction-mapping validation.')
else:
    CIC_CSV = Path(CIC_CSV)
    assert CIC_CSV.is_file(), f'CICFlowMeter CSV not found: {CIC_CSV}'

    # CICFlowMeter must retain these keys; timestamp-only matching is ambiguous.
    cic_header = pd.read_csv(CIC_CSV, nrows=0).columns.str.strip().tolist()
    missing = [column for column in CIC_COLUMNS.values() if column not in cic_header]
    if missing:
        raise ValueError(
            f'CICFlowMeter CSV cannot validate the direction mapping; missing columns: {missing}. '
            'Use an export with endpoint IPs/ports or validate against PCAP-derived flows.'
        )

    # Category 0 is relevant only to these documented endpoints.
    victims = {'172.31.69.24', '172.31.69.13'}
    category_0_destinations = {
        '162.125.3.1', '162.125.3.5', '162.125.3.6', '162.125.248.1',
        '162.125.18.133', '13.58.225.34',
    }

    nf_columns = [
        'FLOW_START_MILLISECONDS', 'IPV4_SRC_ADDR', 'IPV4_DST_ADDR',
        'L4_SRC_PORT', 'L4_DST_PORT', 'IN_BYTES', 'OUT_BYTES',
    ]
    nf_parts = []
    for nf_chunk in pd.read_csv(INPUT_CSVS[0], usecols=nf_columns, chunksize=500_000):
        candidate = (
            nf_chunk['IPV4_SRC_ADDR'].astype(str).isin(victims)
            & nf_chunk['IPV4_DST_ADDR'].astype(str).isin(category_0_destinations)
        )
        nf_parts.append(nf_chunk.loc[candidate])
    nf = pd.concat(nf_parts, ignore_index=True)
    nf['match_time'] = pd.to_datetime(nf['FLOW_START_MILLISECONDS'], unit='ms', utc=True)
    nf = nf.rename(columns={
        'IPV4_SRC_ADDR': 'source_ip', 'IPV4_DST_ADDR': 'destination_ip',
        'L4_SRC_PORT': 'source_port', 'L4_DST_PORT': 'destination_port',
    })

    cic_usecols = list(CIC_COLUMNS.values())
    cic_parts = []
    for cic_chunk in pd.read_csv(CIC_CSV, usecols=cic_usecols, chunksize=500_000):
        cic_chunk.columns = cic_chunk.columns.str.strip()
        candidate = (
            cic_chunk[CIC_COLUMNS['source_ip']].astype(str).isin(victims)
            & cic_chunk[CIC_COLUMNS['destination_ip']].astype(str).isin(category_0_destinations)
        )
        cic_parts.append(cic_chunk.loc[candidate])
    cic = pd.concat(cic_parts, ignore_index=True).rename(columns={value: key for key, value in CIC_COLUMNS.items()})
    cic_time = pd.to_datetime(cic['time'], errors='coerce')
    if getattr(cic_time.dt, 'tz', None) is None:
        cic_time = cic_time.dt.tz_localize(CIC_TIMESTAMP_TIMEZONE)
    cic['match_time'] = cic_time.dt.tz_convert('UTC')
    cic = cic.dropna(subset=['match_time'])

    match_keys = ['source_ip', 'destination_ip', 'source_port', 'destination_port']
    for key in ['source_port', 'destination_port']:
        nf[key] = pd.to_numeric(nf[key], errors='coerce').astype('Int64')
        cic[key] = pd.to_numeric(cic[key], errors='coerce').astype('Int64')

    matched = pd.merge_asof(
        nf.sort_values('match_time'), cic.sort_values('match_time'),
        on='match_time', by=match_keys, direction='nearest',
        tolerance=pd.Timedelta(MATCH_TOLERANCE), suffixes=('_nf', '_cic'),
    ).dropna(subset=['forward_bytes'])

    if matched.empty:
        raise ValueError('No flows matched. Check timestamp timezone, endpoint columns, and MATCH_TOLERANCE.')

    forward = pd.to_numeric(matched['forward_bytes'], errors='coerce')
    incoming = pd.to_numeric(matched['IN_BYTES'], errors='coerce')
    outgoing = pd.to_numeric(matched['OUT_BYTES'], errors='coerce')
    summary = pd.DataFrame({
        'exact_byte_agreement': [(forward == incoming).mean(), (forward == outgoing).mean()],
        'zero_indicator_agreement': [(forward.eq(0) == incoming.eq(0)).mean(), (forward.eq(0) == outgoing.eq(0)).mean()],
    }, index=['IN_BYTES', 'OUT_BYTES'])
    print(f'Matched flows: {len(matched):,}')
    display(summary.sort_values('zero_indicator_agreement', ascending=False))
    print('\nZero/non-zero agreement with Total Length of Fwd Packets:')
    display(pd.crosstab(forward.eq(0), incoming.eq(0), rownames=['Fwd bytes == 0'], colnames=['IN_BYTES == 0']))
    display(pd.crosstab(forward.eq(0), outgoing.eq(0), rownames=['Fwd bytes == 0'], colnames=['OUT_BYTES == 0']))
    print(
        '\nUse FORWARD_BYTES_COLUMN only if one candidate has near-perfect zero-indicator agreement '
        'and the result is stable when MATCH_TOLERANCE is changed (for example, 1s and 3s).'
    )


## Run

Category-4 `Attempted` flows become benign. Category 0 is activated only after validating which NF-v3 column is equivalent to `Total Length of Fwd Packets`; therefore `FORWARD_BYTES_COLUMN` starts as `None`.

In [ ]:
import subprocess

FORWARD_BYTES_COLUMN = None  # E.g. 'OUT_BYTES', only after validation against CSV/PCAP evidence.
command = [
    'python', str(REPO_ROOT / 'code/python/scripts/prepare_nfv3.py'),
    '--output-dir', str(OUTPUT_DIR),
    '--chunksize', '500000',
]
for input_csv in INPUT_CSVS:
    command += ['--input-csv', str(input_csv)]
if FORWARD_BYTES_COLUMN:
    command += ['--forward-bytes-column', FORWARD_BYTES_COLUMN]

print(' '.join(command))
subprocess.run(command, check=True)

In [ ]:
import json

manifest_path = OUTPUT_DIR / 'nfv3_corrected.manifest.json'
manifest = json.loads(manifest_path.read_text())
print(json.dumps(manifest['counts'], indent=2))
print('\nCategory 0:', manifest['category_0_detection'])